# Per-Class Decision Thresholds + Soft Routing

**Extends Stage 6 (Model Improvement).** Builds directly on `two_stage_stageB_ensemble_v2.ipynb`
(current best: α-diverse pool + per-class weight matrix, test macro-F1 **0.6302**).

## The thing to get right first

A per-class multiplier $t_c$ applied to the fused score is **redundant with the weight matrix**:

$$t_c \cdot S(c\mid x) \;=\; t_c \sum_m W[m,c]\,P_m(c\mid x) \;=\; \sum_m \big(t_c W[m,c]\big) P_m(c\mid x)$$

— it is exactly a rescaling of column $c$ of $W$, which coordinate ascent already searched. Row
normalising first does not help either: the normaliser $Z(x)=\sum_c S(c\mid x)$ is constant across
classes, so $\arg\max_c t_c S(c)/Z(x) = \arg\max_c t_c S(c)$. **Thresholding inside the attack block
adds no expressive power.**

It can still help as a *reparameterisation* — column scale is an emergent property of seven greedy
entry-wise moves on a coarse grid, and one direct handle per class is far better conditioned for the
precision↔recall axis. Variant **T-attack** below measures exactly that, and its size is the
empirical test of the argument.

## Where the real headroom is

$W$ has no reach over the **Benign boundary**. The gate makes a hard decision and Stage B never
competes with Benign, so nothing in the ensemble can address the 1,171 Benign flows that leak into
Fuzzers or Exploits' 0.642 recall. **Soft routing** fixes that by composing the two stages into one
distribution:

$$P(\text{Benign}\mid x) = 1 - p_{\text{att}}(x), \qquad
P(k \mid x) = p_{\text{att}}(x)\cdot P_B(k \mid x)$$

Now a threshold on Benign trades against the whole attack block, $p_{\text{att}}$ varies per instance,
and $t_0$ subsumes the gate threshold as a special case. This is **not** absorbable into $W$.

## Protocol

Each parameter set gets its own fitting split, and test is touched once:

| Parameters | Fitted on |
|---|---|
| base pool (7 models) + gate | `train` |
| weight matrix $W$ (7×9) | **DSEL** |
| thresholds $t$ (≤10) | **SEL** |
| final evaluation | **test**, once |

## 1. Rebuild the ensemble

Self-contained: same seeds and splits as every other notebook, so the test set matches row-for-row.
The gate now also exposes `predict_proba`, which soft routing needs.

In [1]:
%pip install xgboost imbalanced-learn -q

import numpy as np, pandas as pd, time, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
RS = 42

X = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([X, y], axis=1).drop_duplicates().reset_index(drop=True)
X, y = df[X.columns], df["Label"]

nun   = X.nunique(); zero = nun[nun <= 1].index.tolist()
corr  = X.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red   = [c for c in upper.columns if (upper[c] >= 0.95).any()]
X = X.drop(columns=list(dict.fromkeys(zero + red)))

X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)
X_tr,  X_val,  y_tr,  y_val  = train_test_split(X_trf, y_trf, test_size=0.2, random_state=RS, stratify=y_trf)
X_dsel, X_sel, y_dsel, y_sel = train_test_split(X_val, y_val, test_size=0.5, random_state=RS, stratify=y_val)

def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw)
    return XGBClassifier(**p)

gate = make_xgb()
gate.fit(X_tr, (y_tr != 0).astype(int),
         sample_weight=compute_sample_weight("balanced", (y_tr != 0).astype(int)))

# p_att = P(attack) -- the quantity soft routing needs
patt_sel  = gate.predict_proba(X_sel)[:, 1]
patt_test = gate.predict_proba(X_test)[:, 1]

attack_labels = sorted(l for l in y_tr.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC  = len(attack_labels)
FAM = [LABELS[idx_to_fam[i]] for i in range(NC)]

def attack_only(Xa, ya):
    m = ya != 0
    return Xa[m].reset_index(drop=True), ya[m].map(fam_to_idx).reset_index(drop=True)

X_tr_att, y_tr_att     = attack_only(X_tr, y_tr)
X_dsel_att, y_dsel_att = attack_only(X_dsel, y_dsel)
print("train attack %d | DSEL attack %d | SEL %d | test %d"
      % (len(y_tr_att), len(y_dsel_att), len(y_sel), len(y_test)))

Note: you may need to restart the kernel to use updated packages.
train attack 51949 | DSEL attack 6494 | SEL 24494 | test 61235


In [2]:
def sample_w(y_arr, wc):
    w = wc[y_arr].astype(float)
    return w * len(w) / w.sum()

def wc_alpha(n, a):
    w = (n.sum() / n) ** a
    return w / w.mean()

n_tr = np.bincount(y_tr_att.values, minlength=NC)
WC25 = wc_alpha(n_tr, 0.25)

t0 = time.time()
POOL = {}
for a in (0.0, 0.25, 0.5, 1.0):
    m = make_xgb(); m.fit(X_tr_att, y_tr_att, sample_weight=sample_w(y_tr_att.values, wc_alpha(n_tr, a)))
    POOL[f"XGB a={a}"] = m
strat = {c: 2000 for c in range(NC) if n_tr[c] < 2000}
Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(X_tr_att, y_tr_att)
m = make_xgb(); m.fit(Xs, ys); POOL["XGB smote2000"] = m
for nm, mdl in (("RF a=0.25", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)),
                ("ET a=0.25", ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS))):
    mdl.fit(X_tr_att, y_tr_att, sample_weight=sample_w(y_tr_att.values, WC25)); POOL[nm] = mdl
MODELS = list(POOL); M = len(MODELS)
print("pool of %d fitted in %.0fs" % (M, time.time() - t0))

def probas_of(Xq):
    return [POOL[k].predict_proba(Xq) for k in MODELS]

P_dsel, P_sel, P_test = probas_of(X_dsel_att), probas_of(X_sel), probas_of(X_test)

# --- refit the per-class weight matrix on DSEL (identical procedure to v2) ---
def fuse(probas, W):
    S = np.zeros_like(probas[0])
    for m_ in range(M):
        S += probas[m_] * W[m_][None, :]
    return S

GRID_W = (0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0)
yd = y_dsel_att.values

def fit_W(probas, ytrue, passes=3):
    W = np.ones((M, NC)) / M
    best = f1_score(ytrue, fuse(probas, W).argmax(1), average="macro", zero_division=0)
    for _ in range(passes):
        for c in range(NC):
            for m_ in range(M):
                cur = W[m_, c]
                for g in GRID_W:
                    W[m_, c] = g
                    if W[:, c].sum() == 0:
                        continue
                    s = f1_score(ytrue, fuse(probas, W).argmax(1), average="macro", zero_division=0)
                    if s > best:
                        best, cur = s, g
                W[m_, c] = cur
    return W, best

W, dsel_mf1 = fit_W(P_dsel, yd)
print("W refitted on DSEL, Stage-B macro-F1 %.4f" % dsel_mf1)

pool of 7 fitted in 61s
W refitted on DSEL, Stage-B macro-F1 0.6278


## 2. Two routing schemes

**Hard** — the current architecture: gate decides Benign, Stage B only speaks for gate-positive rows.

**Soft** — one composed 10-class distribution, $P(0)=1-p_{\text{att}}$ and
$P(k)=p_{\text{att}}\cdot P_B(k)$. Benign now competes in the same argmax, and borderline flows are
weighted by gate confidence instead of being thrown to one side of a hard boundary.

Stage-B scores are row-normalised into a distribution first, so the composition is a genuine
probability product rather than a mix of scales.

In [3]:
def stage_b_proba(probas, W):
    S = fuse(probas, W)
    return S / np.maximum(S.sum(1, keepdims=True), 1e-12)

def tenclass_hard(PB, gate_pred):
    P = np.zeros((len(PB), 10))
    P[gate_pred == 0, 0] = 1.0
    g = gate_pred == 1
    for i in range(NC):
        P[g, idx_to_fam[i]] = PB[g, i]
    return P

def tenclass_soft(PB, p_att):
    P = np.zeros((len(PB), 10))
    P[:, 0] = 1.0 - p_att
    for i in range(NC):
        P[:, idx_to_fam[i]] = p_att * PB[:, i]
    return P

def report(P10, ye, t=None, tag=""):
    pred = (P10 * (np.ones(10) if t is None else t)).argmax(1)
    return dict(variant=tag,
                macro_F1=round(f1_score(ye, pred, average="macro", zero_division=0), 4),
                macro_P =round(precision_score(ye, pred, average="macro", zero_division=0), 4),
                macro_R =round(recall_score(ye, pred, average="macro", zero_division=0), 4),
                acc     =round(float((pred == ye.values).mean()), 4))

PB_sel,  PB_test  = stage_b_proba(P_sel, W),  stage_b_proba(P_test, W)
gate_sel, gate_test = (patt_sel >= 0.5).astype(int), (patt_test >= 0.5).astype(int)

H_sel,  H_test  = tenclass_hard(PB_sel, gate_sel),  tenclass_hard(PB_test, gate_test)
S_sel,  S_test  = tenclass_soft(PB_sel, patt_sel),  tenclass_soft(PB_test, patt_test)

print(pd.DataFrame([report(H_sel, y_sel, tag="hard routing (current, SEL)"),
                    report(S_sel, y_sel, tag="soft routing (SEL)")]).to_string(index=False))

                    variant  macro_F1  macro_P  macro_R    acc
hard routing (current, SEL)    0.6443   0.6762   0.6534 0.9043
         soft routing (SEL)    0.6452   0.6777   0.6527 0.9049


## 3. Fit thresholds on SEL

Coordinate ascent over per-class multipliers, macro-F1 as the objective. Three variants, so the
contributions separate cleanly:

- **T-attack** — thresholds on the 9 attack classes only, hard routing. Predicted to be small: this
  is the reparameterisation of $W$'s column scale, nothing more.
- **T-soft** — thresholds on all 10 classes with soft routing. The genuinely new capability.
- **T-soft (Benign only)** — one parameter, $t_0$. A one-degree-of-freedom control that is exactly
  equivalent to sweeping the gate threshold, included to show how much of any T-soft gain is just the
  Benign boundary moving versus real per-family rebalancing.

In [4]:
GRID_T = (0.1, 0.2, 0.35, 0.5, 0.7, 0.85, 1.0, 1.2, 1.5, 2.0, 3.0, 5.0, 8.0)

def fit_thresholds(P10, ye, classes, passes=4):
    yv = ye.values
    t  = np.ones(10)
    best = f1_score(yv, (P10 * t).argmax(1), average="macro", zero_division=0)
    for _ in range(passes):
        improved = False
        for c in classes:
            cur = t[c]
            for g in GRID_T:
                t[c] = g
                s = f1_score(yv, (P10 * t).argmax(1), average="macro", zero_division=0)
                if s > best + 1e-6:
                    best, cur, improved = s, g, True
            t[c] = cur
        if not improved:
            break
    return t, best

t_att,  s_att  = fit_thresholds(H_sel, y_sel, range(1, 10))
t_soft, s_soft = fit_thresholds(S_sel, y_sel, range(0, 10))
t_ben,  s_ben  = fit_thresholds(S_sel, y_sel, [0])

print("SEL macro-F1 after fitting:")
print("  T-attack (hard, 9 params)      %.4f" % s_att)
print("  T-soft   (soft, 10 params)     %.4f" % s_soft)
print("  T-soft   (soft, Benign only)   %.4f" % s_ben)
print("\nthresholds:")
print(pd.DataFrame({"T-attack": t_att, "T-soft": t_soft, "T-soft(benign only)": t_ben},
                   index=NAMES).to_string())
print("\nimplied gate threshold from t0 (T-soft): p_att > %.4f"
      % (t_soft[0] / (t_soft[0] + 1.0)) if t_soft[0] > 0 else "n/a")

SEL macro-F1 after fitting:
  T-attack (hard, 9 params)      0.6574
  T-soft   (soft, 10 params)     0.6575
  T-soft   (soft, Benign only)   0.6470

thresholds:
                T-attack  T-soft  T-soft(benign only)
Benign               1.0    1.00                  1.2
Analysis             0.5    0.85                  1.0
Backdoor             1.0    0.85                  1.0
DoS                  2.0    1.20                  1.0
Exploits             1.5    1.50                  1.0
Fuzzers              2.0    2.00                  1.0
Generic              0.5    0.70                  1.0
Reconnaissance       1.5    1.00                  1.0
Shellcode            1.2    2.00                  1.0
Worms                1.0    1.00                  1.0

implied gate threshold from t0 (T-soft): p_att > 0.5000


### 3.1 Is T-attack really redundant?

The direct check of the algebra in the header. If $t$ on the attack block is nothing but a rescaling
of $W$'s columns, then folding $t$ into $W$ must reproduce the thresholded predictions **exactly**.

In [5]:
W_folded = W * np.array([t_att[idx_to_fam[i]] for i in range(NC)])[None, :]
pred_thresh = (H_sel * t_att).argmax(1)

PB_folded = stage_b_proba(P_sel, W_folded)   # note: row-normalised, as in the pipeline
pred_folded = tenclass_hard(PB_folded, gate_sel).argmax(1)

agree = (pred_thresh == pred_folded).mean()
print("predictions identical on %.4f%% of SEL rows" % (100 * agree))
print("-> per-class thresholds on the attack block ARE column rescaling of W"
      if agree > 0.9999 else "-> differs (check normalisation path)")

predictions identical on 100.0000% of SEL rows
-> per-class thresholds on the attack block ARE column rescaling of W


## 4. Variant table on SEL

In [6]:
VAR = [
    ("V0 hard routing, no thresholds",     H_sel, None),
    ("V1 soft routing, no thresholds",     S_sel, None),
    ("V2 hard + T-attack (9 params)",      H_sel, t_att),
    ("V3 soft + T-benign only (1 param)",  S_sel, t_ben),
    ("V4 soft + T-all (10 params)",        S_sel, t_soft),
]
sel_tab = pd.DataFrame([report(P, y_sel, t, tag) for tag, P, t in VAR])
print(sel_tab.to_string(index=False))

base = sel_tab.loc[0, "macro_F1"]
print("\ngains over V0 (SEL):")
for i in range(1, len(sel_tab)):
    print("  %-36s %+.4f" % (sel_tab.loc[i, "variant"], sel_tab.loc[i, "macro_F1"] - base))
BEST_I = int(sel_tab["macro_F1"].idxmax())
print("\nselected on SEL: %s" % sel_tab.loc[BEST_I, "variant"])

                          variant  macro_F1  macro_P  macro_R    acc
   V0 hard routing, no thresholds    0.6443   0.6762   0.6534 0.9043
   V1 soft routing, no thresholds    0.6452   0.6777   0.6527 0.9049
    V2 hard + T-attack (9 params)    0.6574   0.7147   0.6394 0.9044
V3 soft + T-benign only (1 param)    0.6470   0.6800   0.6523 0.9051
      V4 soft + T-all (10 params)    0.6575   0.7162   0.6467 0.9055

gains over V0 (SEL):
  V1 soft routing, no thresholds       +0.0009
  V2 hard + T-attack (9 params)        +0.0131
  V3 soft + T-benign only (1 param)    +0.0027
  V4 soft + T-all (10 params)          +0.0132

selected on SEL: V4 soft + T-all (10 params)


## 5. Final evaluation — test set, once

Every variant is shown so the SEL→test transfer is visible, but the **selected** row is the one fixed
on SEL in section 4. Reference: the v2 ensemble at 0.6302.

In [7]:
VAR_TEST = [
    ("V0 hard routing, no thresholds",     H_test, None),
    ("V1 soft routing, no thresholds",     S_test, None),
    ("V2 hard + T-attack (9 params)",      H_test, t_att),
    ("V3 soft + T-benign only (1 param)",  S_test, t_ben),
    ("V4 soft + T-all (10 params)",        S_test, t_soft),
]
test_tab = pd.DataFrame([report(P, y_test, t, tag) for tag, P, t in VAR_TEST])
print(test_tab.to_string(index=False))

sel_name = sel_tab.loc[BEST_I, "variant"]
row = test_tab[test_tab.variant == sel_name].iloc[0]
print("\nSEL-selected variant on TEST: %s -> macro-F1 %.4f" % (sel_name, row["macro_F1"]))
print("vs v2 ensemble (0.6302)       : %+.4f" % (row["macro_F1"] - 0.6302))
print("vs V0 reproduced here (%.4f) : %+.4f"
      % (test_tab.loc[0, "macro_F1"], row["macro_F1"] - test_tab.loc[0, "macro_F1"]))

sel_rank = list(sel_tab.sort_values("macro_F1", ascending=False).variant)
test_rank = list(test_tab.sort_values("macro_F1", ascending=False).variant)
print("\nSEL order : %s" % " > ".join(v.split()[0] for v in sel_rank))
print("test order: %s" % " > ".join(v.split()[0] for v in test_rank))

                          variant  macro_F1  macro_P  macro_R    acc
   V0 hard routing, no thresholds    0.6302   0.6733   0.6330 0.9030
   V1 soft routing, no thresholds    0.6302   0.6758   0.6300 0.9042
    V2 hard + T-attack (9 params)    0.6259   0.7021   0.6037 0.9023
V3 soft + T-benign only (1 param)    0.6301   0.6764   0.6283 0.9045
      V4 soft + T-all (10 params)    0.6232   0.6995   0.6057 0.9039

SEL-selected variant on TEST: V4 soft + T-all (10 params) -> macro-F1 0.6232
vs v2 ensemble (0.6302)       : -0.0070
vs V0 reproduced here (0.6302) : -0.0070

SEL order : V4 > V2 > V3 > V1 > V0
test order: V0 > V1 > V3 > V2 > V4


### 5.1 Per-class effect — did the off-balance classes actually rebalance?

The motivation was Fuzzers at P 0.574 / R 0.926 and Exploits at P 0.900 / R 0.642 — off-balance in
*opposite* directions. If thresholds worked, those two gaps should narrow.

In [8]:
def per_class(P10, ye, t=None):
    pred = (P10 * (np.ones(10) if t is None else t)).argmax(1)
    return pd.DataFrame({
        "P": precision_score(ye, pred, average=None, labels=range(10), zero_division=0),
        "R": recall_score(ye, pred, average=None, labels=range(10), zero_division=0),
        "F1": f1_score(ye, pred, average=None, labels=range(10), zero_division=0)}, index=NAMES)

before = per_class(H_test, y_test)
sel_t  = dict((tag, t) for tag, _, t in VAR_TEST)[sel_name]
sel_P  = dict((tag, P) for tag, P, _ in VAR_TEST)[sel_name]
after  = per_class(sel_P, y_test, sel_t)

cmp = pd.concat([before.add_suffix("_before"), after.add_suffix("_after")], axis=1)
cmp["dF1"] = (after["F1"] - before["F1"]).round(3)
print(cmp.round(3).to_string())

                P_before  R_before  F1_before  P_after  R_after  F1_after    dF1
Benign             0.999     0.968      0.983    0.999    0.967     0.983 -0.000
Analysis           0.318     0.649      0.427    0.338    0.571     0.425 -0.002
Backdoor           0.938     0.500      0.652    0.953    0.456     0.617 -0.036
DoS                0.525     0.368      0.433    0.580    0.348     0.435  0.002
Exploits           0.900     0.642      0.750    0.885    0.668     0.761  0.012
Fuzzers            0.574     0.926      0.708    0.556    0.952     0.702 -0.006
Generic            0.768     0.814      0.790    0.854    0.762     0.805  0.015
Reconnaissance     0.623     0.661      0.641    0.711    0.599     0.651  0.009
Shellcode          0.379     0.343      0.360    0.335    0.359     0.347 -0.013
Worms              0.710     0.458      0.557    0.783    0.375     0.507 -0.050


In [9]:
pred_final = (sel_P * (np.ones(10) if sel_t is None else sel_t)).argmax(1)
print(classification_report(y_test, pred_final, labels=list(range(10)),
                            target_names=NAMES, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_test, pred_final, labels=list(range(10))),
                   index=NAMES, columns=NAMES).to_string())

                precision    recall  f1-score   support

        Benign      0.999     0.967     0.983     45001
      Analysis      0.338     0.571     0.425        77
      Backdoor      0.953     0.456     0.617        90
           DoS      0.580     0.348     0.435       883
      Exploits      0.885     0.668     0.761      6182
       Fuzzers      0.556     0.952     0.702      5279
       Generic      0.854     0.762     0.805       899
Reconnaissance      0.711     0.599     0.651      2406
     Shellcode      0.335     0.359     0.347       370
         Worms      0.783     0.375     0.507        48

      accuracy                          0.904     61235
     macro avg      0.700     0.606     0.623     61235
  weighted avg      0.925     0.904     0.907     61235

                Benign  Analysis  Backdoor  DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
Benign           43530        34         1   12        67     1251       12              70         24 

## 6. Findings

**Negative result: per-class thresholds do not help. Keep V0 — the v2 ensemble's plain argmax, test
macro-F1 0.6302.**

| Variant | Params fitted on SEL | SEL | **Test** |
|---|---|---|---|
| **V0 hard routing, no thresholds** | 0 | 0.6443 | **0.6302** |
| V1 soft routing, no thresholds | 0 | 0.6452 | 0.6302 |
| V3 soft + Benign threshold only | 1 | 0.6470 | 0.6301 |
| V2 hard + T-attack | 9 | 0.6574 | 0.6259 |
| V4 soft + T-all | 10 | **0.6575** | 0.6232 |

The rank order **inverted exactly** between the two splits:

```
SEL  : V4 > V2 > V3 > V1 > V0        (most parameters wins)
test : V0 > V1 > V3 > V2 > V4        (fewest parameters wins)
```

A monotone relationship between parameter count and SEL gain, and the same relationship reversed on
test. That is textbook overfitting, and it is the entire result: every gain on SEL was noise.

### 1. The redundancy claim is confirmed exactly

§3.1 folded the fitted thresholds into the weight matrix and the predictions agreed on
**100.0000% of SEL rows**. The algebra in the header holds at prediction level: per-class thresholds
on the attack block *are* column rescaling of $W$, not a new capability.

This makes V2's +0.0131 on SEL interpretable. It cannot be new expressive power, so it is either
better search conditioning or overfitting — and its −0.0043 on test settles which.

### 2. Soft routing is a no-op, and the gate threshold is already optimal

V1 vs V0: **+0.0009 on SEL, +0.0000 on test** (0.6302 either way). Composing
$P(\text{attack})\cdot P(k\mid\text{attack})$ instead of hard-gating changes essentially nothing.

Stronger still, the threshold search *left Benign at $t_0 = 1.0$* in the full T-soft fit — an implied
gate threshold of exactly $p_{\text{att}} > 0.5$. The one-parameter V3 fit nudged it to 1.2 and gained
+0.0027 on SEL, −0.0001 on test.

**The gate is well calibrated at its default threshold.** That closes a lever that had been on the
next-steps list since the weighting study, and it explains why the Fuzzers false-positive analysis
(34% of them Benign flows leaking through) did not translate into an actionable gain: those leaks are
not threshold-fixable, they are flows the gate genuinely cannot separate.

### 3. Why thresholds had nothing left to do

Two independent degrees of freedom were already spent:

- **Per-class balance** — already optimised by $W$ on DSEL, and proven in §3.1 to be the same
  parameter family. Refitting it on SEL adds variance without adding capacity.
- **The Benign boundary** — the only genuinely new degree of freedom, and already optimal (finding 2).

With both exhausted, the 10 threshold parameters could only fit noise. The per-class table shows
exactly that: the **rare families got worse** (Worms 0.557→0.507, Backdoor 0.652→0.617, Shellcode
0.360→0.347) while well-populated ones improved slightly (Generic +0.015, Exploits +0.012,
Reconnaissance +0.009). SEL supplies only ~19 Worms and ~31 Analysis rows, each worth 1/10 of macro-F1,
so the optimiser chased rare-class noise and paid for it on test. Macro-precision rose 0.673→0.700
while macro-recall fell 0.633→0.606 — a precision drift that looked good on SEL and cost F1 on test.

### 4. The premise behind this experiment was wrong

The motivation was that Fuzzers (P 0.574 / R 0.926) and Exploits (P 0.900 / R 0.642) are "badly
off-balance in opposite directions, near-ideal for threshold tuning." **Asymmetric precision and
recall is not evidence of a suboptimal threshold.** Per-class F1 is not maximised at P ≈ R; when the
two error types have different marginal rates the F1-optimal operating point is genuinely asymmetric,
and $W$ had already found it. Tellingly, the fitted thresholds did not even push consistently toward
balance — Fuzzers became *more* asymmetric (0.556 / 0.952) under a search that was purely maximising
SEL macro-F1.

Worth carrying into the write-up as a methodological point: a lopsided P/R pair is a description of
where a class sits on its trade-off curve, not a diagnosis that it sits in the wrong place.

### 5. Protocol note

$t$ was fitted on SEL, which is also where the v2 architecture was selected. That reuse is mild — the
architecture was already committed — but SEL is no longer clean for further tuning. Any subsequent
hyper-parameter work needs a fresh split carved from `train`.

### Recommendation

**Keep the v2 ensemble as-is (V0, test macro-F1 0.6302).** Add no thresholds, no soft routing, no
gate-threshold tuning — all three are now measured and none of them pays.

The value delivered here is subtractive: three lever candidates that had been sitting on the
next-steps list are now closed with evidence, and the residual error is again shown to be shared
rather than fixable by adjusting the decision rule. That is the same conclusion the focal and DES v1
experiments reached, from a third direction.

### Next levers

With the decision-rule levers exhausted, what remains is either more diversity or a direct measurement
of the ceiling:

1. **Widen the α grid** (1.5, 2.0) and refit the pool on train+DSEL — the only lever with a positive
   track record, since α=1.0 supplied most of the ensemble's diversity.
2. **Label-ambiguity ceiling analysis** — now motivated by three independent negative results all
   pointing at irreducible error rather than model deficiency.
3. **SHAP per-class explainability** — still outstanding in the novelty stack.
